# Step 1a wf_v4 Colab batch (Step 4: schema creation, dialogues-only schemas, placement into an existing schema)
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4`, folder `step1a/`. Contract `WF_V4_BATCH.md`, prompt table `results/wf_v4/colab/manifest_summary.md`. **Stage 1** (`manifest.json`): the first call of 25 chains (4.1 group: inputs a, b, c1, c2 x dev50, dev50b x column-order seeds 0, 1, 2 = 24, plus the placement base chain) and the 6 dialogues-only calls (e). **Stages 2-4** (built by cell 8 with `scripts/batch_stage_v4.py` from the earlier responses): 4.2 name, 4.3 relationships, 4.4 keys and types of every chain that is still valid. **Stages 5 and up:** one placement call per held-out column and mode (with / without the signal card). Then `scripts/assemble_v4.py` writes canonical JSON, SQL and operations per finished chain. Model and settings as v3.9: Qwen2.5-32B-Instruct-AWQ, temperature 0, guided JSON per job schema.

**Before running:** `results/wf_v4/colab/` (stage 1) and `results/wf_v4/inputs/` must be committed **and pushed** on `wf-v4` (`git push -u origin wf-v4`). Runtime > Change runtime type > **A100 (80 GB)**. Colab Secrets `HF_TOKEN` and `GH_TOKEN` with notebook access on.

In [ ]:
#@title 1. Tokens, clone/pull wf-v4, git identity
# Needs Colab Secrets: HF_TOKEN and GH_TOKEN, notebook access on.
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
BRANCH = "wf-v4"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** `pip install vllm` prints dependency-resolver warnings. Ignore them. Then **Runtime > Restart session**, rerun cell 1, skip cell 2.

In [ ]:
#@title 2. Install vLLM (a few minutes; ignore resolver warnings, then restart the session and skip this cell)
!pip install -q vllm

In [ ]:
#@title 3. GPU check (the 32B AWQ model needs >= 38 GiB, i.e. a 40 GB A100)
ALLOW_SMALL_GPU = False  #@param {type:"boolean"}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GB")
# a 40 GB A100 reports about 39.5 GiB
assert gb >= 38 or ALLOW_SMALL_GPU, (
    f"only {gb:.1f} GiB: the 32B model does not fit with a 21K-token prompt. Use an A100, or tick ALLOW_SMALL_GPU "
    "and set MODEL in cell 4 to Qwen/Qwen2.5-14B-Instruct-AWQ")

In [ ]:
#@title 4. Load the model
MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"  #@param ["Qwen/Qwen2.5-32B-Instruct-AWQ", "Qwen/Qwen2.5-14B-Instruct-AWQ"]
REVISION = ""  #@param {type:"string"}
MAX_LEN = 32768
import vllm
from huggingface_hub import model_info
from vllm import LLM, SamplingParams
if not REVISION:
    REVISION = model_info(MODEL).sha  # resolve once, so the cache key names the exact weights
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization="awq",
          max_model_len=MAX_LEN, gpu_memory_utilization=0.92)
tok = llm.get_tokenizer()
MODEL_ID = f"{MODEL}@{REVISION}"  # used in cache keys

In [ ]:
#@title 5. Stage-1 manifest and exact prompt token counts (stops if a first prompt is > 20000 or any prompt > 30000)
import json, sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from pathlib import Path
from s1a import llm_cache as lc
O = Path("results/wf_v4/colab")
LIMIT_FIRST, LIMIT_ANY = 20000, 30000      # exact tokens (first prompts = 4.1 and e)
FAMILY_ORDER = ["s4_41", "s4_42", "s4_43", "s4_44", "s4_e", "s4_place"]
def load_jobs(name="manifest.json"):
    return json.loads((O / name).read_text(encoding="utf-8"))["jobs"]
def chat_prompt(text):
    return tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
prompts, schemas, counts = {}, {}, {}
def prep(job_list):
    """Read prompts and schemas of the jobs, count exact prompt tokens; stop if a first prompt is over LIMIT_FIRST or any over LIMIT_ANY."""
    for j in job_list:
        n = j["job"]
        prompts[n] = (O / j["prompt"]).read_text(encoding="utf-8")
        schemas[n] = json.loads((O / j["schema"]).read_text(encoding="utf-8"))
        counts[n] = len(tok(chat_prompt(prompts[n]), add_special_tokens=False).input_ids)
    if job_list:
        c = [counts[j["job"]] for j in job_list]
        print(len(job_list), "jobs; tokens: min", min(c), "max", max(c), "total", sum(c))
    too_long = {j["job"]: counts[j["job"]] for j in job_list
                if counts[j["job"]] > (LIMIT_FIRST if j["family"] in ("s4_41", "s4_e") else LIMIT_ANY)}
    assert not too_long, f"prompt over its limit: {too_long}; shorten the examples (scripts/inputs_v4.py --examples 3 --width 50) and rebuild"
    return job_list
stage1 = load_jobs()
print(len(stage1), "stage-1 jobs:", {f: sum(1 for j in stage1 if j["family"] == f) for f in FAMILY_ORDER})
prep(stage1)


In [ ]:
#@title 6. Batch generation helper (temperature 0, guided JSON per job schema, unguided retry)
# vLLM >= 0.11: StructuredOutputsParams(json=...) via `structured_outputs`; older: GuidedDecodingParams via `guided_decoding`.
import hashlib, time
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)

def generate_batch(items):
    """(outputs, guided flags) for a list of (job, key, path, params, max_tokens). If guided decoding raises for the
    batch, each job is rerun alone, guided; only a job whose own schema fails runs unguided."""
    texts = [chat_prompt(prompts[it[0]["job"]]) for it in items]
    try:
        return llm.generate(texts, [sampling(it[4], schemas[it[0]["job"]]) for it in items]), [True] * len(items)
    except Exception as e:
        print("  guided decoding failed for the batch, rerunning its jobs one by one:", type(e).__name__, str(e)[:200])
    outs, guided = [], []
    for t, it in zip(texts, items):
        try:
            outs.append(llm.generate([t], sampling(it[4], schemas[it[0]["job"]]))[0]); guided.append(True)
        except Exception as e:
            print(f"  {it[0]['job']}: guided failed ({type(e).__name__}), unguided")
            outs.append(llm.generate([t], sampling(it[4], None))[0]); guided.append(False)
    return outs, guided

def make_row(job, params, prompt, out, guided, tf):
    o = out.outputs[0]
    parsed, errors, _ = lc.parse_job(o.text, job["format"], schemas[job["job"]])
    return {"prompt_sha": hashlib.sha256(prompt.encode("utf-8")).hexdigest(), "model": MODEL, "revision": REVISION,
            "vllm": vllm.__version__, "params": params, "run_tag": job["run_tag"], "job": job["job"],
            "format": job["format"], "guided": guided, "finish_reason": o.finish_reason, "raw": o.text,
            "parsed": parsed, "errors": errors, **tf}

def run_stage(stage_jobs):
    """Run all uncached jobs of `stage_jobs` in batches of BATCH; one cache row per job (keyed by model+revision, params, prompt, run_tag)."""
    todo = []
    for job in stage_jobs:
        name, prompt = job["job"], prompts[job["job"]]
        max_tokens = min(job["max_tokens"], MAX_LEN - counts[name] - 64)
        schema_sha = hashlib.sha256(json.dumps(schemas[name], sort_keys=True).encode("utf-8")).hexdigest()[:16]
        params = {"temperature": 0, "max_tokens": max_tokens, "guided_json": job["schema"], "schema_sha": schema_sha}
        k = lc.key(MODEL_ID, params, prompt, job["run_tag"])
        path = O / "responses" / f"{name}.jsonl"
        if lc.get(path, k) is None:
            todo.append((job, k, path, params, max_tokens))
    print(f"{len(stage_jobs) - len(todo)} cached, {len(todo)} to run in batches of {BATCH}")
    t_stage = time.time()
    for b0 in range(0, len(todo), BATCH):
        items = todo[b0:b0 + BATCH]
        t0 = time.time()
        outs, guided_flags = generate_batch(items)
        secs = time.time() - t0
        for (job, k, path, params, max_tokens), out, guided in zip(items, outs, guided_flags):
            name, prompt = job["job"], prompts[job["job"]]
            row = make_row(job, params, prompt, out, guided, lc.token_fields(out, secs, len(items)))
            if row["errors"] and guided:  # retry once unguided, alone; keep the better one, count both calls' tokens
                t1 = time.time()
                out2 = llm.generate([chat_prompt(prompt)], sampling(max_tokens, None))[0]
                row2 = make_row(job, params, prompt, out2, False, lc.token_fields(out2, time.time() - t1, 1))
                best = row2 if len(row2["errors"]) < len(row["errors"]) else row
                best.update(retried=True, prompt_tokens=row["prompt_tokens"] + row2["prompt_tokens"],
                            output_tokens=row["output_tokens"] + row2["output_tokens"])
                print(f"  retry unguided {name}: errors {len(row['errors'])} -> {len(row2['errors'])}")
                row = best
            lc.put(path, k, row)
            print(f"{name}: tokens in {row['prompt_tokens']} out {row['output_tokens']}, errors {len(row['errors'])}, "
                  f"guided {row['guided']}, finish {row['finish_reason']}", row["errors"][:2])
        done = b0 + len(items)
        print(f"batch of {len(items)} took {secs:.0f} s ({done}/{len(todo)} done)")
        if b0 == 0 and done < len(todo):
            est = secs / len(items) * (len(todo) - done) / 60
            print(f"rough time for the remaining {len(todo) - done} jobs: {est:.0f} min (later batches may be faster)")
    print(f"stage elapsed {(time.time() - t_stage) / 60:.1f} min")


In [ ]:
#@title 7. Stage 1: the first call of every chain and the dialogues-only calls
run_stage(stage1)

In [ ]:
#@title 8. Stages 2 to 4 and the placement stages: build each stage from the responses of the earlier ones, then run it
# scripts/batch_stage_v4.py reads the responses, validates every chain's answers (a chain stops at its first missing, unparsable or
# invalid answer; see chain_status_stage<k>.json) and writes manifest_stage<k>.json with the prompts of the next calls.
import subprocess
N_PLACE = len(json.loads((O / "placement.json").read_text(encoding="utf-8"))["steps"])
all_stages = {1: stage1}
for k in range(2, 5 + N_PLACE):
    r = subprocess.run(["python", "scripts/batch_stage_v4.py", "--dir", str(O), "--stage", str(k)], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-800:])
    assert r.returncode == 0, r.stderr[-800:]
    all_stages[k] = load_jobs(f"manifest_stage{k}.json")
    if not all_stages[k]:
        print(f"stage {k} is empty: every chain stopped"); break
    prep(all_stages[k])
    run_stage(all_stages[k])
print({k: len(v) for k, v in all_stages.items()})

In [ ]:
#@title 9. Assemble: canonical JSON, SQL and operations of every finished chain
!python scripts/assemble_v4.py --dir results/wf_v4/colab

In [ ]:
#@title 10. Parse check: every job of every stage has a response row without parse errors
all_jobs = [j for v in all_stages.values() for j in v]
bad, missing = {}, []
for j in all_jobs:
    path = O / "responses" / f"{j['job']}.jsonl"
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []
    if not rows:
        missing.append(j["job"]); continue
    parsed, errors, _ = lc.parse_job(rows[-1]["raw"], j["format"], schemas[j["job"]])
    if errors:
        bad[j["job"]] = errors[:2]
print(len(all_jobs), "jobs;", len(missing), "without a response;", len(bad), "with parse errors")
for k, v in list(bad.items())[:20]:
    print(" ", k, v)
final = json.loads((O / "final" / "status.json").read_text(encoding="utf-8"))
stopped = [r for r in final if not r["ok"]]
print(len(final), "runs,", len(stopped), "not assembled:", [(r["run"], r["error"]) for r in stopped[:5]])
assert not missing, missing[:5]

In [ ]:
#@title 10. Summary: calls, tokens and wall time per job family
import collections
kind_of = {j["job"]: j["family"] for j in all_jobs}
agg = collections.defaultdict(lambda: [0, 0, 0, 0.0, 0])  # calls, prompt tokens, output tokens, wall s, rows without token fields
for name, kind in kind_of.items():
    path = O / "responses" / f"{name}.jsonl"
    if not path.exists():
        continue
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not rows:
        continue
    r, a = rows[-1], agg[kind]
    a[0] += 1
    if "prompt_tokens" in r:
        a[1] += r["prompt_tokens"]; a[2] += r["output_tokens"]; a[3] += r["batch_seconds"] / r["batch_size"]
    else:
        a[4] += 1
print(f"{'family':10} {'calls':>6} {'prompt tok':>11} {'output tok':>11} {'wall min':>9}")
for kind in FAMILY_ORDER:
    if kind in agg:
        c, p, o, s, miss = agg[kind]
        print(f"{kind:10} {c:6d} {p:11d} {o:11d} {s / 60:9.1f}" + (f"  ({miss} rows without token fields)" if miss else ""))
tot = [sum(a[i] for a in agg.values()) for i in range(4)]
print(f"{'total':10} {tot[0]:6d} {tot[1]:11d} {tot[2]:11d} {tot[3] / 60:9.1f}   (wall = sum of batch_seconds / batch_size over rows)")


In [ ]:
#@title 11. Commit and push results/wf_v4/colab to wf-v4
!bash colab/sync.sh push-results "v4 batch responses" wf_v4/colab